In [ ]:
!uv pip install geocoder

In [ ]:
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
import os
from contextlib import AsyncExitStack
from agents import Agent, Runner, trace
import gradio as gr

In [ ]:
load_dotenv(override=True)

In [ ]:
weather_params = {
  "command": "npx",
  "args": ["-y", "-p", "open-meteo-mcp-server", "open-meteo-mcp-server"],
  "env": {
    "OPEN_METEO_API_URL": "https://api.open-meteo.com",
    "OPEN_METEO_AIR_QUALITY_API_URL": "https://air-quality-api.open-meteo.com",
    "OPEN_METEO_MARINE_API_URL": "https://marine-api.open-meteo.com",
    "OPEN_METEO_ARCHIVE_API_URL": "https://archive-api.open-meteo.com",
    "OPEN_METEO_SEASONAL_API_URL": "https://seasonal-api.open-meteo.com",
    "OPEN_METEO_ENSEMBLE_API_URL": "https://ensemble-api.open-meteo.com",
    "OPEN_METEO_GEOCODING_API_URL": "https://geocoding-api.open-meteo.com",
    "OPEN_METEO_FLOOD_API_URL": "https://flood-api.open-meteo.com",
    "OPEN_METEO_CLIMATE_API_URL": "https://climate-api.open-meteo.com"
  }
}

geo_params = {"command": "uv", "args": ["run", "geo-server.py"]}

brave_params = {
  "command": "npx",
  "args": [
    "-y",
    "@brave/brave-search-mcp-server"
  ],
  "env": {
    "BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")
  }
}

calendar_params = {
  "command": "npx",
  "args": ["@cocal/google-calendar-mcp"],
  "env": {
    "GOOGLE_OAUTH_CREDENTIALS": "/Users/macbookpro/Projects/agents/6_mcp/community_contributions/elikeyz/gcp-oauth.keys.json"
  }
}

mcp_server_params = [weather_params, geo_params, brave_params, calendar_params]

In [ ]:
async with AsyncExitStack() as stack:
  weather_server, geo_server, brave_server, calendar_server = [await stack.enter_async_context(MCPServerStdio(params, client_session_timeout_seconds=30)) for params in mcp_server_params]
  weather_tools = await weather_server.list_tools()
  geo_tools = await geo_server.list_tools()
  brave_tools = await brave_server.list_tools()
  calendar_tools = await calendar_server.list_tools()

In [ ]:
print("Weather Tools:")
print(weather_tools)
print("\nGeo Tools:")
print(geo_tools)
print("\nCalendar Tools:")
print(calendar_tools)
print("\nBrave Tools:")
print(brave_tools)

In [ ]:
name = "Elijah Enuem-Udogu"
instructions = f"""
My name is {name}
You are my personal assistant.
You can use the following tools:
- Weather: Get the weather in a specific location
- Geocoder: Get the latitude and longitude of a specific location
- Brave: Search the web for local and international news
- Calendar: Get, create, update, delete and respond to events in my calendar and get the current date and time.

**IMPORTANT**: When using the calendar, if the user is not authenticated, you can trigger authentication using the manage-accounts add tool. This should be done by default if there are not authenticated accounts.

You need to use the geocoder tool to get my location, then check the weather for that location using the weather tool.

You must provide me with the top latest local and international news when I ask for the news.

When searching for local news, you need to use the geocoder tool to get my location, then use my location by default except I say otherwise.

Whenever I ask you to check the weather, you need to use the geocoder tool to get my location, then use my location by default except I say otherwise.

Also, don't mention the coordinates when responding, instead mention the name of the city and country. If you are unsure of the city, just mention the name of the city in the approximate location. Also, say the city name instead of saying "your location" or "my location".

You must always use the calendar tool to get the current date and time and give accurate responses based on the current date and time.

Use this information to help me to organize my day. Inform me of the weather and how it may affect any events I have coming up.
"""
model = "gpt-4.1-mini"

In [ ]:
async def chatbot(prompt, history):
    try:
        async with AsyncExitStack() as stack:
            mcp_servers = [await stack.enter_async_context(MCPServerStdio(params, client_session_timeout_seconds=30)) for params in mcp_server_params]
            agent = Agent(name="personal-assistant", instructions=instructions, model=model, mcp_servers=mcp_servers)
            with trace("personal-assistant"):
                result = await Runner.run(agent, prompt)
                return result.final_output
    except Exception as e:
        return str(e)

In [ ]:
gr.ChatInterface(chatbot).launch()